In [1]:
from XGBoostModel import XGBoostModel
from RunData import RunPipeline
import pandas as pd
import optuna
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold ,train_test_split, cross_val_score
import kagglehub
import os
import zipfile
from seed_utils import SEED
import requests


/home/daniel7/.conda/envs/my_env_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# GenericDataPipeline

In [2]:
class GenericDataPipeline:

    def rank_features(self, X, y, n_folds=5):
        print(f"Starting feature importance calculation with XGBoost and {n_folds}-fold CV...")

        X_copy = X.copy()

        null_ratios = {}
        for col in X_copy.columns:
            null_ratios[col] = X_copy[col].isna().mean()

        for col in X_copy.select_dtypes(include=['int64', 'float64']).columns:
            X_copy[col] = X_copy[col].replace([np.inf, -np.inf], np.nan)
            
            non_null = X_copy[col].dropna()
            if len(non_null) > 0:
                mean_val = non_null.mean()
                std_val = non_null.std()
                if std_val > 0 and not np.isnan(mean_val):
                    upper_bound = mean_val + 5 * std_val
                    lower_bound = mean_val - 5 * std_val
                    X_copy[col] = X_copy[col].clip(lower=lower_bound, upper=upper_bound)
        
        num_cols = X_copy.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = X_copy.select_dtypes(include=['object']).columns
        
        X_processed = X_copy.copy()
        
        for col in cat_cols:
            if X_processed[col].isna().sum() > 0:
                most_freq = X_processed[col].mode()[0] if not X_processed[col].isna().all() else 'Unknown'
                X_processed[col] = X_processed[col].fillna(most_freq)
            X_processed[col] = X_processed[col].astype('category')
        
        for col in num_cols:
            if X_processed[col].isna().sum() > 0:
                median_val = X_processed[col].median() if not X_processed[col].isna().all() else 0
                X_processed[col] = X_processed[col].fillna(median_val)
        
        X_filled = X_processed

        mean_target = float(np.mean(y))
        print(f"Mean target value (for base_score): {mean_target:.6f}")

        base_score = 0.5
        if 0 < mean_target < 1:
            base_score = mean_target

        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'max_depth': 6,
            'eta': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 3,
            'alpha': 0.01,
            'lambda': 1.0,
            'gamma': 0.1,
            'base_score': base_score, 
            'seed': 42,
            'tree_method': 'hist',
            'device': 'cuda'

        }

        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

        fold_importances = []

        fold_scores = []

        print(f"Training XGBoost models across {n_folds} folds...")

        y_values = set(y.unique())
        print(f"Unique target values: {y_values}")

        if not all(isinstance(val, (int, float)) for val in y_values):
            print("Warning: Target variable contains non-numeric values. Converting to numeric.")
            y = y.astype(float)

        if not all(val in [0, 1] for val in y_values):
            print("Warning: Target variable contains values other than 0 and 1.")
            print("Converting target to binary: 0 for negative class, 1 for positive class.")
            y = (y > 0).astype(int)

        # Perform cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_filled, y)):
            print(f"Fold {fold+1}/{n_folds}")

            X_train, X_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            train_pos = np.mean(y_train)
            val_pos = np.mean(y_val)
            print(f"  Train positive rate: {train_pos:.4f}, Validation positive rate: {val_pos:.4f}")

            try:
                dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
                dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

                # Train model
                model = xgb.train(
                    params,
                    dtrain,
                    num_boost_round=100,
                    early_stopping_rounds=20,
                    evals=[(dval, 'validation')],
                    verbose_eval=False
                )

                y_pred = model.predict(dval)
                score = roc_auc_score(y_val, y_pred)
                fold_scores.append(score)

                try:
                    importance = model.get_score(importance_type='gain')

                    fold_importance = {col: importance.get(col, 0) for col in X_filled.columns}

                    max_value = max(fold_importance.values()) if max(fold_importance.values()) > 0 else 1
                    normalized_importance = {col: val/max_value for col, val in fold_importance.items()}

                    fold_importances.append(normalized_importance)
                except Exception as e:
                    print(f"Warning: Could not get feature importance for fold {fold+1}: {str(e)}")

            except Exception as e:
                print(f"Error in fold {fold+1}: {str(e)}")
                print("Skipping this fold and continuing with others.")

        if fold_scores:
            mean_auc = np.mean(fold_scores)
            print(f"Mean AUC across {len(fold_scores)} successful folds: {mean_auc:.4f}")
        else:
            print("No successful folds. Cannot calculate mean AUC.")

        if not fold_importances:
            print("No feature importance information could be gathered from any fold.")
            print("Using a simple fallback method for feature scoring.")

            avg_importance = {}
            for col in X_filled.columns:
                try:
                    valid_mask = ~pd.isna(X_copy[col]) & ~pd.isna(y)
                    if valid_mask.sum() > 10: 
                        if X_filled[col].dtype.name == 'category':
                            col_values = X_filled[col][valid_mask].cat.codes
                        else:
                            col_values = X_filled[col][valid_mask]
                        
                        y_values = y[valid_mask]
                        corr = abs(np.corrcoef(col_values, y_values)[0, 1])
                        avg_importance[col] = corr if not np.isnan(corr) else 0
                    else:
                        avg_importance[col] = np.random.uniform(0, 0.001)
                except Exception as e:
                    print(f"Warning: Could not calculate correlation for {col}: {str(e)}")
                    avg_importance[col] = np.random.uniform(0, 0.001)
        else:
            avg_importance = {}
            for col in X_filled.columns:
                importances = [fold_imp.get(col, 0) for fold_imp in fold_importances]
                if importances:
                    avg_importance[col] = np.mean(importances)
                else:
                    null_ratio = null_ratios.get(col, 0)
                    avg_importance[col] = max(0.001 * (1 - null_ratio), 0.0001)  

        feature_data = []
        for col in X_copy.columns:
            gain_score = avg_importance.get(col, 0)
            null_ratio = null_ratios.get(col, 0)
            feature_data.append({
                'feature_name': col,
                'gain_score': gain_score,
                'null_ratio': null_ratio
            })
        
        feature_df = pd.DataFrame(feature_data)
        feature_df = feature_df.sort_values('gain_score', ascending=False)
        
        print("\nFeature scores:")
        print("-" * 100)
        print(f"{'Rank':<5} | {'Feature':<30} | {'Gain Score':<15} | {'Null Ratio':<10}")
        print("-" * 100)

        for i, row in feature_df.iterrows():
            rank = i + 1
            feat = row['feature_name']
            gain = row['gain_score']
            null_ratio = row['null_ratio']
            print(f"{rank:<5} | {feat:<30} | {gain:.6f} | {null_ratio:.4f}")

        return feature_df

    def objective(self, trial, feature_scores_with_nulls, all_features_score, df, label="readmitted"):
        K = 10
        selected_features = feature_scores_with_nulls['feature_name'].to_list()[:K]
        print("selected_features")
        print(selected_features)
        all_features = all_features_score['feature_name'].to_list()
        # Create binary assignment for each of the selected features:
        assignment = {}
        for feat in selected_features:
            # 1 means feature goes to stage1, 0 means feature is used in stage2
            assignment[feat] = trial.suggest_categorical(f"assign_{feat}", [1, 0])
        
        for feat in all_features:
            if feat not in assignment.keys():
                assignment[feat] = 1
        vals = 0
        for key,value in assignment.items():
            vals +=value
        if vals==0:
            return 99999
        penalty = 0 
        # Derive feature sets based on the assignment:
        stage1_features = [feat for feat, assign in assignment.items() if assign == 1]
        stage2_features = [feat for feat, assign in assignment.items() if assign == 0]
        csv_name = f"trial_{trial.number}_results.csv"
        dm = RunPipeline()
        validation_loss = dm.full_run(df,stage1_features,stage2_features,label,csv_name)
        if validation_loss == 999:
            return validation_loss
        else:
            final_objective = validation_loss + penalty
            return final_objective
    def preprocessing(self,df):
        for col in df.columns:
            if df[col].dtype == 'object':
                # Replace ? with NaN
                df[col] = df[col].replace(['?', ''], np.nan)
                # Convert to categorical codes
                if df[col].isna().sum() < len(df):  # If not all values are NaN
                    df[col] = pd.Categorical(df[col]).codes

                    # Ensure -1 (missing) values are converted to NaN
                    df[col] = df[col].replace(-1, np.nan)
        # Convert boolean columns to int
        for col in df.select_dtypes(include=['bool']).columns:
            df[col] = df[col].astype(int)
        return df 
    def Whether_client(self,n_trials=20):
        path = kagglehub.dataset_download("shilongzhuang/telecom-customer-churn-by-maven-analytics")

        csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

        csv_path1 = os.path.join(path, csv_files[0])
        csv_path2 = os.path.join(path, csv_files[1])
        csv_path3 = os.path.join(path, csv_files[2])

        data1 = pd.read_csv(csv_path1, encoding='latin1')
        data2 = pd.read_csv(csv_path3, encoding='latin1')
        # columns_to_drop  = ['dataset','id','ca', 'thal', 'slope','age']
        df = pd.merge(data2, data1, on='Zip Code')
        df['Customer Status'] = df['Customer Status'].apply(lambda x: 1 if x == 'Stayed' else 0)
        columns_to_drop  = ['Customer ID','Churn Category','Churn Reason',
                            'Total Charges','Total Revenue','Total Refunds',
                           'Total Long Distance Charges','Zip Code','City','Latitude','Longitude']
        df = df.drop(columns=columns_to_drop)
        df = self.preprocessing(df)
        return df

# [Whether the client appeared on the record](https://www.kaggle.com/datasets/kapturovalexander/whether-the-client-appeared-on-the-record)

In [3]:
fs = GenericDataPipeline()
study = fs.Whether_client(20)


Using 'Customer Status' as target variable
Starting feature importance calculation with XGBoost and 5-fold CV...
Mean target value (for base_score): 0.670169
Training XGBoost models across 5 folds...
Unique target values: {np.int64(0), np.int64(1)}
Fold 1/5
  Train positive rate: 0.6702, Validation positive rate: 0.6700
Fold 2/5
  Train positive rate: 0.6702, Validation positive rate: 0.6700
Fold 3/5
  Train positive rate: 0.6702, Validation positive rate: 0.6700
Fold 4/5
  Train positive rate: 0.6701, Validation positive rate: 0.6705
Fold 5/5
  Train positive rate: 0.6701, Validation positive rate: 0.6705


[I 2025-05-17 11:12:31,104] A new study created in memory with name: no-name-a44ba6a2-356b-40ad-9f90-b2c3a0d29310
[I 2025-05-17 11:12:31,137] A new study created in memory with name: no-name-15d7d462-3788-4823-8d4f-c28dae3b14fb


Mean AUC across 5 successful folds: 0.9497

Feature scores:
----------------------------------------------------------------------------------------------------
Rank  | Feature                        | Gain Score      | Null Ratio
----------------------------------------------------------------------------------------------------
23    | Contract                       | 1.000000 | 0.0000
7     | Tenure in Months               | 0.665682 | 0.0000
5     | Number of Dependents           | 0.208074 | 0.0000
6     | Number of Referrals            | 0.197269 | 0.0000
12    | Internet Service               | 0.173423 | 0.0000
26    | Monthly Charge                 | 0.107474 | 0.0000
3     | Age                            | 0.104909 | 0.0000
20    | Streaming Movies               | 0.099155 | 0.2167
8     | Offer                          | 0.096827 | 0.5505
18    | Premium Tech Support           | 0.091981 | 0.2167
4     | Married                        | 0.086236 | 0.0000
15    | Online Secu

/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
/home/daniel7/.conda/envs/my_env_311/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [11:12:31] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1742531989799/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
[I 2025-

********** run results **********
{'max_depth': 4, 'learning_rate': 0.002712728156907108, 'n_estimators': 386, 'min_child_weight': 3, 'subsample': 0.909523655293813, 'colsample_bytree': 0.9033869070968912, 'gamma': 0.36277585606310614, 'lambda': 8.410036352821077, 'alpha': 7.714033948085811, 'scale_pos_weight': 0.8577802387903852, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.9176740638113848), np.float64(0.9387940978804932), np.float64(0.9118794699728165)]
********** run results **********


...

********** run results **********
{'max_depth': 6, 'learning_rate': 0.019369760635011488, 'n_estimators': 868, 'min_child_weight': 3, 'subsample': 0.83745260982884, 'colsample_bytree': 0.8294076853414944, 'gamma': 2.0110353382298984, 'lambda': 1.1195821287506482, 'alpha': 6.010240152401851, 'scale_pos_weight': 9.181972879254344, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.9451380657974822), np.float64(0.9450974491689488), np.float64(0.9432397460202822)]
********** run results **********


/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
[I 2025-05-17 11:57:46,250] Trial 19 finished with value: 0.9414629336795143 and parameters: {'max_depth': 4, 'learning_rate': 0.0070425168689511285, 'n_estimators': 479, 'min_child_weight': 6, 'subsample': 0.9989074536845045, 'colsample_bytree': 0.8918132176038761, 'gamma': 0.684981748876865, 'lambda': 3.279912219856446, 'alpha': 4.056747981541062, 'scale_pos_weight': 6.552520752445973}. Best is trial 13 with value: 0.9219929241912457.


********** run results **********
{'max_depth': 4, 'learning_rate': 0.0070425168689511285, 'n_estimators': 479, 'min_child_weight': 6, 'subsample': 0.9989074536845045, 'colsample_bytree': 0.8918132176038761, 'gamma': 0.684981748876865, 'lambda': 3.279912219856446, 'alpha': 4.056747981541062, 'scale_pos_weight': 6.552520752445973, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.9404531184931287), np.float64(0.9425526365943446), np.float64(0.9413830459510698)]
********** run results **********
Best parameters found: {'max_depth': 4, 'learning_rate': 0.0030299080160693967, 'n_estimators': 351, 'min_child_weight': 1, 'subsample': 0.8426698001414014, 'colsample_bytree': 0.9371109464146444, 'gamma': 0.003687898041607074, 'lambda': 1.7942662002449108, 'alpha': 4.957444285808622, 'scale_pos_weight': 9.748783152212095}
Best CV AUC score: 0.9220

===== Detailed Cross-Validation Results with Best Parameters =====
Params: {'max_depth': 4, 'lea

[I 2025-05-17 11:57:47,584] Trial 19 finished with value: -0.0003625920780470082 and parameters: {'assign_Streaming Movies': 1, 'assign_Offer': 0, 'assign_Premium Tech Support': 0, 'assign_Online Security': 1, 'assign_Streaming TV': 0, 'assign_Internet Type': 0, 'assign_Unlimited Data': 0, 'assign_Streaming Music': 0, 'assign_Avg Monthly GB Download': 1, 'assign_Online Backup': 0}. Best is trial 14 with value: -0.03064417627516136.


Test set AUC (with extended features): 0.9108
Test set AUC (without extended features): 0.9665
Overall test set AUC: 0.9199
Streaming Movies: 0.0200
Online Security: 0.0392
Avg Monthly GB Download: 0.0160
Contract: 0.0930
Tenure in Months: 0.5646
Number of Dependents: 0.0194
Number of Referrals: 0.0316
Internet Service: 0.0000
Monthly Charge: 0.0249
Age: 0.0690
Married: 0.0000
Phone Service: 0.0028
Payment Method: 0.0004
Paperless Billing: 0.0000
Total Extra Data Charges: 0.0008
Population: 0.0044
Multiple Lines: 0.0027
Avg Monthly Long Distance Charges: 0.0033
Device Protection Plan: 0.0000
Gender: 0.0001
Offer: 0.0318
Premium Tech Support: 0.0385
Streaming TV: 0.0000
Internet Type: 0.0181
Unlimited Data: 0.0031
Streaming Music: 0.0000
Online Backup: 0.0163
has_extended: 0.0000

Top 10 features by importance:
Tenure in Months: 0.5646
Contract: 0.0930
Age: 0.0690
Online Security: 0.0392
Premium Tech Support: 0.0385
Offer: 0.0318
Number of Referrals: 0.0316
Monthly Charge: 0.0249
Stream

# Final Results

AUC Summary:
                         Model      AUC
      Base model (no extended) 0.971648
Extended model (with extended) 0.920247
  Combined model (no extended) 0.969891
Combined model (with extended) 0.914718

Base Features:
Offer, Streaming TV, Internet Type, Unlimited Data, Streaming Music, Avg Monthly GB Download, Online Backup, Contract, Tenure in Months, Number of Dependents, Number of Referrals, Internet Service, Monthly Charge, Age, Married, Phone Service, Payment Method, Paperless Billing, Total Extra Data Charges, Population, Multiple Lines, Avg Monthly Long Distance Charges, Device Protection Plan, Gender

Extended Features:
Streaming Movies, Premium Tech Support, Online Security, has_extended

Target - Customer Status

AUC Differences:
Extended model - Combined (with extended): 0.005529000000000006
Base model - Combined (no extended):       0.0017570000000000086

Standard Deviations:
Extended model - Std Dev: 0.0064
Base model - Std Dev:     0.0134
